In [1]:
# @title Cell 0: Install Required Packages
# Run this cell first if you haven't already in your session.
!pip install -q selenium webdriver-manager beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 7.4 MB/s eta 0:00:00


In [2]:
# @title Cell 1: Install Chrome, Selenium & WebDriver, Mount Drive, Core Imports
# This cell is largely the same as your original Cell 1.
# It sets up Chrome, Selenium, WebDriver, and performs essential imports.

# ###########################################################################
# #################### VERY IMPORTANT PREREQUISITES #########################
# ###########################################################################
#
# 1. Before running THIS cell (Cell 1), you MUST run the following command
#    in a COMPLETELY SEPARATE Colab code cell (like Cell 0 above):
#
#    !pip install -q selenium webdriver-manager beautifulsoup4 requests
#
#    After that command finishes successfully in the separate cell,
#    THEN you can proceed with this cell.
#
# 2. This cell will now also attempt to install Google Chrome by first
#    adding its official repository.
#
# ###########################################################################

print("--- Cell 1: Initial Setup & Imports ---")
print("--- Setting up Google Chrome Repository and Installing Chrome ---")
# Download and add Google's Linux package signing key
!wget -q -O - https://dl.google.com/linux/linux_signing_key.pub | apt-key add -
# Add the Google Chrome repository to sources.list
!grep -q "dl.google.com/linux/chrome/deb" /etc/apt/sources.list.d/google-chrome.list || sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'


print("\n--- Updating package lists (after adding Chrome repo) ---")
!apt-get update -qq
print("--- Package list update attempt complete ---")

print("\n--- Installing Google Chrome (required for Selenium) ---")
# Install Google Chrome stable
!apt-get install -y google-chrome-stable
print("--- Google Chrome installation attempt complete ---")

print("\n--- Verifying Chrome installation (Shell commands) ---")
!which google-chrome-stable
!google-chrome-stable --version
!ls -l /usr/bin/google-chrome-stable
print("--- Chrome installation verification (Shell commands) complete ---")

print("\n--- Python Imports and Setup ---")
import os
import time
import re
import concurrent.futures # Added for parallelization

# Python check for Chrome binary
CHROME_BINARY_PATH = "/usr/bin/google-chrome-stable"
print(f"--- Verifying Chrome installation (Python check for {CHROME_BINARY_PATH}) ---")
if os.path.exists(CHROME_BINARY_PATH):
    print(f"SUCCESS: Chrome binary found at {CHROME_BINARY_PATH} by Python's os.path.exists().")
    if os.access(CHROME_BINARY_PATH, os.X_OK):
        print(f"SUCCESS: Chrome binary at {CHROME_BINARY_PATH} is executable by Python's os.access().")
    else:
        print(f"WARNING: Chrome binary found at {CHROME_BINARY_PATH}, but it does NOT appear to be executable by Python's os.access(). Check permissions.")
else:
    print(f"ERROR: Chrome binary NOT found at {CHROME_BINARY_PATH} by Python's os.path.exists(). This is likely the cause of the 'cannot find Chrome binary' error in Selenium.")
    print("       This suggests the 'apt-get install google-chrome-stable' command did not place the binary at the expected location or failed, even after adding the repository.")
print("--- Chrome installation verification (Python check) complete ---")


# Selenium imports
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service as ChromeService
    from selenium.webdriver.common.by import By
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    print("\n'selenium' library modules imported successfully.")
except ImportError:
    print("------------------------------------------------------------------------------------")
    print("IMPORT ERROR: The 'selenium' library is NOT INSTALLED or NOT FOUND.")
    print("\n>>> IMMEDIATE ACTION REQUIRED TO FIX THIS ERROR: <<<")
    print("1. STOP execution of this cell.")
    print("2. In a NEW, SEPARATE Colab cell, type EXACTLY: !pip install -q selenium webdriver-manager")
    print("3. Run that NEW cell and wait for it to complete.")
    print("4. CRITICAL: After successful installation, re-run THIS cell (Cell 1).")
    print("------------------------------------------------------------------------------------")
    raise

try:
    from webdriver_manager.chrome import ChromeDriverManager
    print("'webdriver_manager.chrome' imported successfully.")
except ImportError:
    print("------------------------------------------------------------------------------------")
    print("IMPORT ERROR: The 'webdriver_manager' library is NOT INSTALLED or NOT FOUND.")
    print("\n>>> IMMEDIATE ACTION REQUIRED TO FIX THIS ERROR: <<<")
    print("1. Ensure you have successfully run in a separate cell: !pip install -q selenium webdriver-manager")
    print("2. CRITICAL: After successful installation, re-run THIS cell (Cell 1).")
    print("------------------------------------------------------------------------------------")
    raise

# BeautifulSoup
try:
    from bs4 import BeautifulSoup
    print("'BeautifulSoup' (from bs4) imported successfully.")
except ImportError:
    print("---------------------------------------------------------------------------")
    print("IMPORT ERROR: The 'BeautifulSoup4' (bs4) library is not found.")
    print("\n>>> ACTION REQUIRED TO FIX THIS ERROR: <<<")
    print("1. In a NEW, SEPARATE Colab cell, type EXACTLY: !pip install beautifulsoup4")
    print("2. Run that NEW cell and wait for it to complete.")
    print("3. CRITICAL: After successful installation, re-run THIS cell (Cell 1).")
    print("---------------------------------------------------------------------------")
    raise

# Requests (for HTTP downloads)
try:
    import requests
    print("'requests' imported successfully.")
except ImportError:
    print("---------------------------------------------------------------------------")
    print("IMPORT ERROR: The 'requests' library is not found.")
    print("\n>>> ACTION REQUIRED TO FIX THIS ERROR: <<<")
    print("1. In a NEW, SEPARATE Colab cell, type EXACTLY: !pip install requests")
    print("2. Run that NEW cell and wait for it to complete.")
    print("3. CRITICAL: After successful installation, re-run THIS cell (Cell 1).")
    print("---------------------------------------------------------------------------")
    raise


# Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True) # force_remount can help with stale mounts
    print("Google Drive mounted successfully at /content/drive.")
except ImportError:
    print("NOTE: 'google.colab.drive' not found. Assuming not in Colab or Drive not needed.")
except Exception as e:
    print(f"ERROR: Could not mount Google Drive: {e}")

# --- WebDriver Setup Function (Global settings accessed here) ---
# @markdown Optional: Disable image loading in Selenium for potentially faster page loads.
DISABLE_IMAGE_LOADING = True # @param {type:"boolean"}

def setup_driver(headless=True): # Added headless parameter
    """Sets up and returns a Selenium WebDriver instance."""
    # This function will now be called by individual threads for link collection,
    # and once for the initial main page scan.
    # Print statements here might interleave if multiple threads call it near-simultaneously.
    # For parallel use, consider making it quieter or using a lock for prints if noisy.
    # For now, we'll keep them but be aware of potential interleaved output during parallel setup.

    # print("\nSetting up Chrome WebDriver...") # Quieter for threaded calls
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")

    # Access global DISABLE_IMAGE_LOADING defined in this cell's scope
    if DISABLE_IMAGE_LOADING:
        # print("  Attempting to disable image loading.")
        chrome_options.add_argument('--blink-settings=imagesEnabled=false')
        # print("  Image loading disabled in Chrome options.")


    # Explicitly set the binary location for Chrome
    # Access global CHROME_BINARY_PATH defined in this cell's scope
    chrome_options.binary_location = CHROME_BINARY_PATH

    # Use a common user agent
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")

    try:
        # print("Attempting to install/update ChromeDriver via webdriver-manager...")
        service = ChromeService(ChromeDriverManager().install())
        driver_instance = webdriver.Chrome(service=service, options=chrome_options)
        # print("WebDriver setup complete.")
        return driver_instance
    except Exception as e:
        print(f"ERROR: WebDriver setup failed: {e}")
        print(f"Ensure Google Chrome is installed and accessible at '{CHROME_BINARY_PATH}'. The Python check earlier should confirm this.")
        print("Ensure that your Colab environment has internet access to download ChromeDriver.")
        return None

print("\nCell 1 execution complete. Chrome setup, core imports, and WebDriver setup function defined.")

--- Cell 1: Initial Setup & Imports ---
--- Setting up Google Chrome Repository and Installing Chrome ---
OK
grep: /etc/apt/sources.list.d/google-chrome.list: No such file or directory

--- Updating package lists (after adding Chrome repo) ---
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--- Package list update attempt complete ---

--- Installing Google Chrome (required for Selenium) ---
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libvulkan1 mesa-vulkan-drivers
The following NEW packages will be installed:
  google-chrome-stable libvulkan1 mesa-vulkan-d

In [31]:
# @title Cell 2: Configuration
# This cell is largely the same as your original Cell 2.
# Added MAX_WORKERS for parallelization.

print("--- Cell 2: Configuration ---")

# --- User Configuration ---
# @markdown The web address to the Addgene list of plasmids.
TARGET_URL = "https://www.addgene.org/browse/article/28196835/" # @param {"type":"string"}

# @markdown The path on your Google Drive where the .gbk files should be stored.
DRIVE_DOWNLOAD_PATH = "/content/drive/My Drive/Addgene_Lavickova2019Maerkl" # @param {"type":"string"}

# @markdown Maximum number of parallel workers for collecting plasmid links (Selenium tasks). Adjust based on your Colab instance's resources and politeness to Addgene.
MAX_WORKERS_LINK_COLLECTION = 12  # @param {type:"integer"}

# @markdown Maximum number of parallel workers for downloading files (HTTP tasks).
MAX_WORKERS_DOWNLOADS = 8  # @param {type:"integer"}

# @markdown Delay between individual file downloads (if desired, though parallel downloads inherently space out requests).
DOWNLOAD_ITEM_DELAY_SECONDS = 1 # @param {type:"integer"}

PAGE_LOAD_TIMEOUT_SECONDS = 60 # Max time to wait for a page to load with Selenium
ELEMENT_LOAD_TIMEOUT_SECONDS = 30 # Max time to wait for specific elements to appear

# @markdown Maximum number of plasmids to process from the main list (0 for all). Useful for testing.
MAX_PLASMIDS_TO_PROCESS_OVERALL = 0 # @param {type:"integer"}


BASE_ADDGENE_URL = "https://www.addgene.org"
# --- End User Configuration ---

# Create download directory
try:
    if not os.path.exists(DRIVE_DOWNLOAD_PATH):
        os.makedirs(DRIVE_DOWNLOAD_PATH)
        print(f"Successfully created directory: {DRIVE_DOWNLOAD_PATH}")
    else:
        print(f"Directory already exists: {DRIVE_DOWNLOAD_PATH}")
except Exception as e:
    print(f"ERROR creating directory {DRIVE_DOWNLOAD_PATH}: {e}")

print(f"\nTarget URL for scraping: {TARGET_URL}")
print(f"Files will be downloaded to: {DRIVE_DOWNLOAD_PATH}")
print(f"Max workers for link collection: {MAX_WORKERS_LINK_COLLECTION}")
print(f"Max workers for downloads: {MAX_WORKERS_DOWNLOADS}")
print(f"Max plasmids to process: {'All' if MAX_PLASMIDS_TO_PROCESS_OVERALL == 0 else MAX_PLASMIDS_TO_PROCESS_OVERALL}")


# Store configuration in a dictionary to pass to worker functions
# This helps avoid relying on global variables within threaded functions, making them more robust.
# CHROME_BINARY_PATH and DISABLE_IMAGE_LOADING are used by setup_driver and are global in Cell 1.
# If setup_driver were in a different module, these would also need to be passed.
# For notebook structure, direct access from setup_driver (in Cell 1) to its globals is fine.
CONFIG = {
    "DRIVE_DOWNLOAD_PATH": DRIVE_DOWNLOAD_PATH,
    "BASE_ADDGENE_URL": BASE_ADDGENE_URL,
    "ELEMENT_LOAD_TIMEOUT_SECONDS": ELEMENT_LOAD_TIMEOUT_SECONDS,
    "PAGE_LOAD_TIMEOUT_SECONDS": PAGE_LOAD_TIMEOUT_SECONDS,
    # CHROME_BINARY_PATH and DISABLE_IMAGE_LOADING are accessed globally by setup_driver in Cell 1
}


print("\nCell 2 execution complete. Configuration set.")

--- Cell 2: Configuration ---
Successfully created directory: /content/drive/My Drive/Addgene_Lavickova2019Maerkl

Target URL for scraping: https://www.addgene.org/browse/article/28196835/
Files will be downloaded to: /content/drive/My Drive/Addgene_Lavickova2019Maerkl
Max workers for link collection: 12
Max workers for downloads: 8
Max plasmids to process: All

Cell 2 execution complete. Configuration set.


In [4]:
# @title Cell 3: Helper Functions (Main Page Parsing, HTTP Download)
# This cell contains find_plasmid_data_on_main_page_selenium and download_file_http.
# find_plasmid_data_on_main_page_selenium is modified to accept a driver instance.
# CORRECTED: The lambda function in WebDriverWait for table update.

# Imports from Cell 1 (os, re, time, BeautifulSoup, requests, Selenium specifics) should be available.
from selenium.webdriver.support.ui import Select # For interacting with <select> dropdowns
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
import traceback

print("--- Cell 3: Helper Functions (Corrected) ---")

def fetch_page_content_selenium(driver, url, wait_for_element_locator=None, save_html_path=None, initial_load=False, config=None):
    """
    Fetches the HTML content of a given URL using an existing Selenium driver.
    Optionally waits for a specific element to be present.
    Optionally saves the fetched HTML to a file for inspection.
    """
    if not driver:
        print("  ERROR: Selenium WebDriver is not provided in fetch_page_content_selenium.")
        return None
    if not config:
        print("  ERROR: Config object not provided to fetch_page_content_selenium.")
        return None

    element_load_timeout = config.get("ELEMENT_LOAD_TIMEOUT_SECONDS", 30)

    try:
        if initial_load:
             print(f"  Navigating to {url} with Selenium...")
        driver.get(url)

        if wait_for_element_locator:
            try:
                print(f"  Waiting up to {element_load_timeout}s for element {wait_for_element_locator} to be present...")
                WebDriverWait(driver, element_load_timeout).until(
                    EC.presence_of_element_located(wait_for_element_locator)
                )
                print(f"  Element {wait_for_element_locator} found.")
            except TimeoutException:
                print(f"  WARNING: Timed out waiting for element {wait_for_element_locator} on {url}. Proceeding with current page source.")
            except Exception as e:
                print(f"  WARNING: Error waiting for element {wait_for_element_locator} on {url}. Proceeding. Error: {e}")
        else:
            # print(f"  No specific element to wait for, adding a small fixed delay for {url}...")
            time.sleep(0.5) # Reduced general delay, main wait should be for specific elements

        html_content = driver.page_source

        if save_html_path:
            try:
                with open(save_html_path, 'w', encoding='utf-8') as f:
                    f.write(html_content)
                print(f"  DEBUG: Successfully saved Selenium-fetched HTML to: {save_html_path}")
            except Exception as e:
                print(f"  DEBUG: Error saving Selenium-fetched HTML to {save_html_path}: {e}")

        return html_content
    except Exception as e:
        print(f"  ERROR: Selenium failed to get URL {url}: {e}")
        return None

def find_plasmid_data_on_main_page_selenium(driver, main_page_url, config):
    """
    Finds plasmid data (URL, name, ID) from a PI or article summary page using Selenium.
    Accepts a driver instance and config dictionary.
    Adapts to different table IDs/structures. Attempts to set table to show all entries more dynamically.
    Returns:
        list: A list of dictionaries, each containing 'url', 'name', 'id'.
    """
    print(f"\nSearching for plasmid data on (Selenium): {main_page_url}")

    drive_download_path = config["DRIVE_DOWNLOAD_PATH"]
    base_addgene_url = config["BASE_ADDGENE_URL"]
    element_load_timeout = config["ELEMENT_LOAD_TIMEOUT_SECONDS"]

    page_type_slug = "unknown_page"
    if "/pi/" in main_page_url or "David_Liu" in main_page_url: page_type_slug = "pi_page"
    elif "/browse/article/" in main_page_url: page_type_slug = "article_page"

    html_save_filename = f"addgene_selenium_main_{page_type_slug}_vLatest_parallel.html"
    html_full_save_path = os.path.join(drive_download_path, html_save_filename) if drive_download_path else html_save_filename

    print(f"  Navigating to {main_page_url} with Selenium for initial table load...")
    driver.get(main_page_url) # Use the passed driver

    table_id_to_find = None
    table_actual_id_for_parsing = None
    table_length_dropdown_name = None
    table_found_by_type = "None"
    found_table_element = None
    link_column_index = -1
    name_column_index = -1

    possible_table_locators = [
        {"id": "plasmids-datatable", "name_attr_for_length": "plasmids-datatable_length", "type": "PI Page Table", "link_col_idx": 1, "name_col_idx": 2},
        {"id": "DataTables_Table_0", "name_attr_for_length": "DataTables_Table_0_length", "type": "Article Page Table (Generic DataTable ID)", "link_col_idx": 0, "name_col_idx": 1},
        {"id": "plasmid-table-publication", "name_attr_for_length": "plasmid-table-publication_length", "type": "Article Page Table (Old ID)", "link_col_idx": 1, "name_col_idx": 1}
    ]

    for loc_info in possible_table_locators:
        try:
            locator = (By.ID, loc_info["id"])
            print(f"  Attempting to find table by ID: {loc_info['id']} ({loc_info['type']})")
            WebDriverWait(driver, 7).until(EC.presence_of_element_located(locator))
            content_locator = (By.CSS_SELECTOR, f"#{loc_info['id']} tbody tr")
            print(f"  Table ID '{loc_info['id']}' found. Waiting for its content (tbody tr)...")
            WebDriverWait(driver, element_load_timeout).until(EC.presence_of_element_located(content_locator))
            table_id_to_find = loc_info["id"]
            table_actual_id_for_parsing = loc_info["id"]
            found_table_element = driver.find_element(*locator)
            table_length_dropdown_name = loc_info["name_attr_for_length"]
            table_found_by_type = loc_info["type"]
            link_column_index = loc_info["link_col_idx"]
            name_column_index = loc_info["name_col_idx"]
            print(f"  Successfully located table and its content via: {table_found_by_type} (ID: '{table_id_to_find}', Link Col: {link_column_index}, Name Col: {name_column_index})")
            break
        except TimeoutException:
            print(f"  Table or its content not found using: {loc_info['type']} (ID: {loc_info['id']}) within specified timeout.")
            continue
        except Exception as e_find_table:
            print(f"  Error finding table {loc_info['id']}: {e_find_table}")
            continue


    if not found_table_element:
        print(f"  ERROR: Could not locate a known plasmid data table on {main_page_url} using predefined locators.")
        if html_full_save_path:
            try:
                with open(html_full_save_path, 'w', encoding='utf-8') as f: f.write(driver.page_source)
                print(f"  DEBUG: Saved current HTML to: {html_full_save_path} for inspection.")
            except Exception as e_save_html:
                 print(f"  DEBUG: Failed to save HTML for table not found: {e_save_html}")
        return []

    initial_row_count = len(driver.find_elements(By.CSS_SELECTOR, f"#{table_id_to_find} tbody tr"))
    print(f"  Initial number of rows in '{table_id_to_find}': {initial_row_count}")

    if table_length_dropdown_name:
        try:
            select_element_check = Select(driver.find_element(By.NAME, table_length_dropdown_name))
            all_option_present = any(option.get_attribute("value") == "-1" for option in select_element_check.options)

            if all_option_present and initial_row_count < 25: # Adjust threshold as needed
                print(f"  Attempting to set table '{table_id_to_find}' to 'Show All' entries using dropdown name '{table_length_dropdown_name}'...")
                show_entries_dropdown_locator = (By.NAME, table_length_dropdown_name)
                WebDriverWait(driver, element_load_timeout).until(EC.element_to_be_clickable(show_entries_dropdown_locator))

                select_element = Select(driver.find_element(*show_entries_dropdown_locator))
                select_element.select_by_value("-1")
                print(f"  Selected 'All' entries for '{table_id_to_find}'. Waiting for table to reload/update (max 10s)...")
                time.sleep(1) # Give a brief moment for JS to start

                try:
                    # CORRECTED LAMBDA FUNCTION with implicit line continuation
                    WebDriverWait(driver, 10).until(
                        lambda d: (
                            len(d.find_elements(By.CSS_SELECTOR, f"#{table_id_to_find} tbody tr")) > initial_row_count or
                            (
                                len(d.find_elements(By.ID, f"{table_id_to_find}_processing")) > 0 and
                                d.find_element(By.ID, f"{table_id_to_find}_processing").value_of_css_property('display') == 'none'
                            ) or
                            len(d.find_elements(By.ID, f"{table_id_to_find}_processing")) == 0
                        )
                    )
                except TimeoutException:
                    print("  WARNING: Timed out waiting for table to fully update after 'Show All'. Proceeding with current state.")
                except StaleElementReferenceException:
                    print("  INFO: Table reloaded, StaleElementReferenceException caught, which is often normal after table update. Proceeding.")
                    time.sleep(2) # Brief pause after stale element

                current_rows_after_show_all = driver.find_elements(By.CSS_SELECTOR, f"#{table_id_to_find} tbody tr")
                print(f"  Number of rows in '{table_id_to_find}' after 'Show All' attempt: {len(current_rows_after_show_all)}")
                if len(current_rows_after_show_all) <= initial_row_count and initial_row_count < 100 :
                     print(f"  WARNING: Row count did not significantly increase after 'Show All'.")
            else:
                print(f"  INFO: 'Show All' option not present, not needed for current row count ({initial_row_count}), or already showing many. Skipping 'Show All'.")

        except Exception as e:
            print(f"  WARNING: Could not set table '{table_id_to_find}' to 'Show All'. Error: {e}. Proceeding with default view.")
    else:
        print(f"  INFO: No specific table_length_dropdown_name identified for pagination. Skipping pagination attempt.")

    print("  Fetching page source after pagination attempt...")
    content = driver.page_source
    if html_full_save_path:
        try:
            with open(html_full_save_path, 'w', encoding='utf-8') as f: f.write(content)
            print(f"  DEBUG: Successfully saved Selenium-fetched HTML (after pagination) to: {html_full_save_path}")
        except Exception as e:
            print(f"  DEBUG: Error saving Selenium-fetched HTML to {html_full_save_path}: {e}")

    if not content:
        print(f"  ERROR: Selenium content for {main_page_url} is None after pagination. Cannot find plasmid links.")
        return []

    print(f"  DEBUG: Selenium-fetched content length (after pagination) for {main_page_url}: {len(content)}")

    soup = BeautifulSoup(content, 'html.parser')
    plasmid_data_list = []
    extracted_count = 0
    target_table_soup = soup.find('table', id=table_actual_id_for_parsing)

    if target_table_soup:
        print(f"  DEBUG (Selenium): Processing table identified by BeautifulSoup as ID '{table_actual_id_for_parsing}'.")
        rows = target_table_soup.select('tbody tr')
        print(f"  DEBUG: Found {len(rows)} rows in table tbody for parsing.")
        plasmid_link_regex = re.compile(r"^/(\d+)/?$") # Ensure it captures the ID for Addgene links

        for row_idx, row in enumerate(rows):
            cells = row.find_all('td')
            plasmid_name = "UnknownPlasmidName"
            plasmid_id = "UnknownID"
            href_str = None

            if len(cells) > max(link_column_index, name_column_index):
                name_cell = cells[name_column_index]
                name_a_tag = name_cell.find('a')
                if name_a_tag and name_a_tag.string: plasmid_name = str(name_a_tag.string).strip()
                elif name_cell.string: plasmid_name = str(name_cell.string).strip()
                # Fallback for name if it's inside the link tag itself and link_column_index == name_column_index
                elif link_column_index == name_column_index and name_a_tag and name_a_tag.string :
                    plasmid_name = str(name_a_tag.string).strip()


                link_cell = cells[link_column_index]
                link_a_tag = link_cell.find('a', href=True)

                if link_a_tag:
                    href_original = link_a_tag.get('href')
                    if href_original:
                        href_str = str(href_original).strip()
                        match_object = plasmid_link_regex.fullmatch(href_str)
                        if match_object:
                            id_from_href = match_object.group(1) # Get captured ID
                            if id_from_href: plasmid_id = id_from_href
                            full_url = base_addgene_url + href_str
                            if not full_url.endswith('/'): full_url += '/' # Normalize URL

                            # If name wasn't found from its dedicated column, try to get it from link text
                            if plasmid_name == "UnknownPlasmidName" and link_a_tag.string:
                                plasmid_name = str(link_a_tag.string).strip()

                            if not any(d['url'] == full_url for d in plasmid_data_list):
                                plasmid_data_list.append({'url': full_url, 'name': plasmid_name, 'id': plasmid_id})
                                extracted_count += 1
        if extracted_count > 0:
            print(f"\n  INFO (Selenium): Extracted {extracted_count} unique plasmid data items from table '{table_actual_id_for_parsing}'.")
        else:
            print(f"\n  WARNING (Selenium): No plasmid links extracted from table '{table_actual_id_for_parsing}' matching pattern '{plasmid_link_regex.pattern}'.")
    else:
        print(f"  ERROR (Selenium): Could not re-locate the identified table with ID '{table_actual_id_for_parsing}' in the parsed HTML soup.")

    if not plasmid_data_list: # Fallback search
        print("\n  INFO (Selenium): Targeted table search failed or yielded no links. Performing general fallback link search on entire page...")
        general_search_count = 0
        plasmid_link_regex_fb = re.compile(r"^/(\d+)/?$")
        for a_tag_fallback in soup.find_all('a', href=plasmid_link_regex_fb):
            href_fallback = str(a_tag_fallback['href']).strip()
            full_url = base_addgene_url + href_fallback
            if not full_url.endswith('/'): full_url += '/'

            id_match_fb = plasmid_link_regex_fb.fullmatch(href_fallback)
            plasmid_id_fb = id_match_fb.group(1) if id_match_fb else "UnknownID_fb"

            name_fb = str(a_tag_fallback.string).strip() if a_tag_fallback.string else f"Plasmid_{plasmid_id_fb}"
            if not any(d['url'] == full_url for d in plasmid_data_list):
                plasmid_data_list.append({'url': full_url, 'name': name_fb, 'id': plasmid_id_fb})
                general_search_count +=1
        if general_search_count > 0:
            print(f"  DEBUG (Selenium): General fallback search found {general_search_count} plasmid links.")
            if extracted_count == 0: # Only update extracted_count if primary method failed
                 extracted_count = general_search_count
                 print(f"\n  INFO (Selenium): Extracted {extracted_count} unique plasmid data items via fallback search.")

    if not plasmid_data_list:
        print(f"\n  WARNING (Selenium): No plasmid page links found on {main_page_url} even with Selenium after all attempts.")
        if html_full_save_path:
             print(f"           Please inspect the saved HTML: {html_full_save_path}")
    return plasmid_data_list


def download_file_http(url, folder_path, plasmid_name="UnknownPlasmid", plasmid_id="0", sequence_id="0", retries=3, delay=5):
    """Downloads a file from a URL using requests and saves it with custom naming."""
    # This function is generally thread-safe as requests sessions are usually managed per request.
    for attempt in range(retries):
        try:
            # print(f"    Attempting HTTP download (Attempt {attempt + 1}/{retries}): {url}") # Can be noisy in parallel
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }
            response = requests.get(url, headers=headers, stream=True, timeout=60, allow_redirects=True)
            response.raise_for_status() # Raises an HTTPError for bad responses (4XX or 5XX)

            # Sanitize plasmid_name for filename
            safe_plasmid_name = re.sub(r'[<>:\\\"/\\\\|?*()]', '', plasmid_name).replace(' ', '_').strip('_')
            if not safe_plasmid_name: safe_plasmid_name = "UnnamedPlasmid" # Handle empty names after sanitization

            plasmid_id_to_use = str(plasmid_id).strip()
            sequence_id_to_use = str(sequence_id).strip()

            final_file_name = f"{safe_plasmid_name}-{plasmid_id_to_use}-{sequence_id_to_use}.gbk"
            # Further sanitize the whole filename (though previous step covers most)
            final_file_name = re.sub(r'[<>:\\\"/\\\\|?*]', '_', final_file_name).replace(' ', '_')
            final_file_name = final_file_name.replace("__", "_") # Clean up double underscores

            file_path = os.path.join(folder_path, final_file_name)
            # print(f"        Attempting to save as: {file_path}")

            with open(file_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"    SUCCESS Download: {final_file_name} (from {url})")
            return file_path
        except requests.exceptions.RequestException as e:
            print(f"    ERROR downloading (HTTP) {url} (Attempt {attempt + 1}): {e}")
            if attempt < retries - 1:
                print(f"    Retrying download in {delay} seconds...")
                time.sleep(delay)
            else:
                print(f"    FAILED to download {url} after {retries} attempts.")
        except IOError as e:
            fn_local = final_file_name if 'final_file_name' in locals() else 'unknown_filename'
            print(f"    IOError saving file (HTTP) {fn_local}: {e}")
            break # Don't retry on IOError, likely a persistent path/permission issue
        except Exception as e:
            print(f"    An unexpected error occurred in download_file_http for {url}: {e}")
            traceback.print_exc()
            break # Don't retry on unexpected errors
    return None

print("Cell 3 execution complete. Helper functions defined (with correction).")

--- Cell 3: Helper Functions (Corrected) ---
Cell 3 execution complete. Helper functions defined (with correction).


In [5]:
# @title Cell 4: Core Scraping Logic (Individual Plasmid Page - For Parallel Execution) - Revised
# This cell contains get_genbank_download_link_from_plasmid_page_selenium_threaded.
# REVISED: Link finding logic reverted closer to original, common_ancestor check removed, limit removed.
# Added some targeted diagnostic prints.

# Imports from Cell 1 (os, re, BeautifulSoup, time, Selenium specifics) should be available.
# setup_driver function from Cell 1 is also needed.
import traceback
import threading # Added in original Cell 4 for thread ID

print("--- Cell 4: Core Scraping Logic for Individual Plasmid Pages (Threaded) - Revised ---")

def get_genbank_download_link_from_plasmid_page_selenium_threaded(plasmid_info, config):
    """
    Worker function to find GenBank download link for a single plasmid.
    Creates and quits its own Selenium driver.
    Args:
        plasmid_info (dict): Dict containing 'url', 'name', 'id' of the plasmid.
        config (dict): Dictionary with configuration like DRIVE_DOWNLOAD_PATH, BASE_ADDGENE_URL, etc.
    Returns:
        dict: Containing 'download_url', 'plasmid_name', 'plasmid_id', 'sequence_id', or None if not found/error.
    """
    plasmid_page_url = plasmid_info['url']
    plasmid_name_from_main_page = plasmid_info['name']
    plasmid_id_from_main_page = plasmid_info['id']

    drive_download_path = config["DRIVE_DOWNLOAD_PATH"]
    base_addgene_url = config["BASE_ADDGENE_URL"]
    element_load_timeout = config["ELEMENT_LOAD_TIMEOUT_SECONDS"]

    thread_id_obj = threading.get_ident()
    thread_id = str(thread_id_obj)[-5:] # Shorter thread ID for logging
    # Initial status print
    print(f"    [T-{thread_id}] START: {plasmid_id_from_main_page} ({plasmid_page_url})")

    driver = None
    try:
        driver = setup_driver() # setup_driver is from Cell 1
        if not driver:
            print(f"      [T-{thread_id}] ERROR: WebDriver setup failed for plasmid {plasmid_id_from_main_page}.")
            return None

        plasmid_id_for_filename = plasmid_id_from_main_page if plasmid_id_from_main_page != "UnknownID" else "unknown_plasmid"

        driver.get(plasmid_page_url)
        WebDriverWait(driver, element_load_timeout).until(
            EC.presence_of_element_located((By.TAG_NAME, "h1"))
        )

        sequences_link_element = None
        sequences_link_locators = [
            (By.CSS_SELECTOR, f"a[href='/{plasmid_id_for_filename}/sequences/']"),
            (By.XPATH, f"//a[contains(@href, '/{plasmid_id_for_filename}/sequences/') and (contains(normalize-space(.), 'Sequence') or contains(normalize-space(.), 'sequence'))]"),
            (By.PARTIAL_LINK_TEXT, "Sequences")
        ]

        for i, locator in enumerate(sequences_link_locators):
            try:
                sequences_link_element = WebDriverWait(driver, element_load_timeout if i < 2 else 10).until(
                    EC.element_to_be_clickable(locator)
                )
                href_attr = sequences_link_element.get_attribute('href')
                if locator[0] == By.PARTIAL_LINK_TEXT:
                    if not href_attr or f"/{plasmid_id_for_filename}/sequences/" not in href_attr:
                        sequences_link_element = None
                        continue
                # print(f"      [T-{thread_id}] Found 'Sequences' link using {locator[0]} for {plasmid_id_for_filename}")
                break
            except TimeoutException:
                continue
            except Exception as e_find_link:
                print(f"      [T-{thread_id}] Non-Timeout ERROR finding sequences link with {locator} for {plasmid_id_for_filename}: {e_find_link}")
                continue

        if not sequences_link_element:
            print(f"      [T-{thread_id}] ERROR: Could not find 'Sequences' link on {plasmid_page_url} for {plasmid_id_for_filename} after all attempts.")
            # Consider saving page source here for debugging if this error is common
            # with open(os.path.join(drive_download_path, f"debug_{plasmid_id_for_filename}_no_sequences_link.html"), "w", encoding="utf-8") as f_debug:
            #    f_debug.write(driver.page_source)
            return None

        sequences_page_url_actual = sequences_link_element.get_attribute('href')
        sequences_link_element.click()
        WebDriverWait(driver, element_load_timeout).until(EC.url_contains("/sequences/"))
        WebDriverWait(driver, element_load_timeout).until(EC.presence_of_element_located((By.TAG_NAME, "h3")))
        time.sleep(1) # Grace period

        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'html.parser')
        # Optional: Save sequences page HTML for debugging failed parses
        # seq_page_debug_path = os.path.join(drive_download_path, f"debug_{plasmid_id_for_filename}_sequences_page.html")
        # with open(seq_page_debug_path, "w", encoding="utf-8") as f_debug_seq:
        #    f_debug_seq.write(page_source)


        genbank_link_element_bs = None
        full_sequence_heading_element = None
        heading_texts_to_find = ["Full Sequence", "Full Sequences from Addgene", "Depositor-provided full sequence"]
        for heading_text_candidate in heading_texts_to_find:
            headings = soup.find_all(['h2', 'h3', 'h4', 'h5', 'strong'],
                                     string=re.compile(r"^\s*" + re.escape(heading_text_candidate), re.IGNORECASE))
            if headings:
                full_sequence_heading_element = headings[0]
                # print(f"      [T-{thread_id}] Found heading: '{full_sequence_heading_element.get_text(strip=True)}' for {plasmid_id_for_filename}")
                break

        if full_sequence_heading_element:
            possible_links = full_sequence_heading_element.find_all_next('a') # Removed limit
            # print(f"      [T-{thread_id}] {plasmid_id_for_filename}: Found {len(possible_links)} links after heading '{full_sequence_heading_element.get_text(strip=True)}'")
            for i_link, link_candidate in enumerate(possible_links):
                href = link_candidate.get('href', '')
                link_text_raw = link_candidate.get_text()
                link_text_stripped = link_text_raw.strip().lower()

                cond1_valid_href = "/genbank/" in href.lower() or href.lower().endswith(".gbk") # Original simple check
                if not cond1_valid_href:
                    # if i_link < 3: print(f"        [T-{thread_id}] Link {i_link} skipped (href invalid): {href[:50]}")
                    continue

                cond2_text_contains_genbank = re.search(r"GenBank", link_text_raw, re.IGNORECASE) is not None
                if not cond2_text_contains_genbank:
                    # if i_link < 3: print(f"        [T-{thread_id}] Link {i_link} skipped (no 'GenBank' in text): {link_text_raw[:50]}")
                    continue

                cond3_is_analyze_sequence = "analyze sequence" in link_text_stripped # Original simple check
                if cond3_is_analyze_sequence:
                    # if i_link < 3: print(f"        [T-{thread_id}] Link {i_link} skipped (is 'analyze sequence'): {link_text_raw[:50]}")
                    continue

                # Removed common_ancestor_level check for simplicity, back to original behavior
                genbank_link_element_bs = link_candidate
                # print(f"      [T-{thread_id}] SUCCESS (Priority): Found GenBank link for {plasmid_id_for_filename}. Text: '{link_text_raw.strip()[:50]}', Href: {href[:70]}")
                break
            # if not genbank_link_element_bs and len(possible_links)>0:
            #     print(f"      [T-{thread_id}] {plasmid_id_for_filename}: No link after heading met all criteria.")

        if not genbank_link_element_bs:
            # print(f"      [T-{thread_id}] FALLBACK: Searching for any GenBank link on page for {plasmid_id_for_filename}...")
            # Original simple fallback regex, ensure \. is used for literal dot.
            all_genbank_links_on_page = soup.find_all('a', href=re.compile(r'(/genbank/|\.gbk$)', re.IGNORECASE))
            candidate_fallback_links = []
            # print(f"        [T-{thread_id}] {plasmid_id_for_filename}: Found {len(all_genbank_links_on_page)} raw fallback links.")
            for link in all_genbank_links_on_page:
                href_fb = link.get('href', '')
                link_text_raw_fb = link.get_text()
                link_text_stripped_fb = link_text_raw_fb.strip().lower()

                # Using same conditions as original successful fallback
                cond1_fb_text_contains_genbank = re.search(r"GenBank", link_text_raw_fb, re.IGNORECASE) is not None
                cond2_fb_is_analyze_sequence = "analyze sequence" in link_text_stripped_fb

                if cond1_fb_text_contains_genbank and not cond2_fb_is_analyze_sequence:
                    candidate_fallback_links.append(link)

            if candidate_fallback_links:
                genbank_link_element_bs = candidate_fallback_links[0] # Simplest: take the first valid one
                # print(f"      [T-{thread_id}] SUCCESS (Fallback): Found GenBank link for {plasmid_id_for_filename}. Text: '{genbank_link_element_bs.get_text(strip=True)[:50]}', Href: {genbank_link_element_bs.get('href')[:70]}")
            # else:
                # print(f"      [T-{thread_id}] {plasmid_id_for_filename}: No suitable fallback links found.")


        if genbank_link_element_bs:
            genbank_download_url = genbank_link_element_bs.get('href')
            if genbank_download_url.startswith('//'):
                 genbank_download_url = "https:" + genbank_download_url
            elif genbank_download_url.startswith('/'):
                genbank_download_url = base_addgene_url + genbank_download_url

            sequence_id = "UnknownSeqID"
            match_path = re.search(r'/sequences?/(\d+)/', genbank_download_url)
            # Simpler regex for .gbk from URL as in original script
            match_fname_gbk = re.search(r'sequence[_-]?(\d+).*\.gbk', genbank_download_url, re.IGNORECASE)

            if match_path: sequence_id = match_path.group(1)
            elif match_fname_gbk: sequence_id = match_fname_gbk.group(1)
            else:
                link_text_check = genbank_link_element_bs.get_text(strip=True)
                match_text = re.search(r'Sequence\s*#?(\d+)', link_text_check, re.IGNORECASE)
                if match_text: sequence_id = match_text.group(1)

            print(f"    [T-{thread_id}] SUCCESS LINK: Plasmid {plasmid_id_from_main_page}, SeqID {sequence_id}, URL: ...{genbank_download_url[-70:]}")
            return {
                'download_url': genbank_download_url,
                'plasmid_name': plasmid_name_from_main_page,
                'plasmid_id': plasmid_id_from_main_page,
                'sequence_id': sequence_id
            }
        else:
            print(f"      [T-{thread_id}] ERROR: No GenBank download link element found for {plasmid_id_for_filename} after all parsing attempts.")
            # Save HTML for final failure analysis if this message is reached:
            # final_fail_path = os.path.join(drive_download_path, f"debug_{plasmid_id_for_filename}_final_fail_sequences_page.html")
            # try:
            #     with open(final_fail_path, "w", encoding="utf-8") as f_debug_final:
            #        f_debug_final.write(page_source if 'page_source' in locals() else "No page source available")
            #     print(f"      [T-{thread_id}] Saved debug HTML for final fail: {final_fail_path}")
            # except Exception as e_save_final_fail:
            #     print(f"      [T-{thread_id}] Could not save final fail HTML: {e_save_final_fail}")

            return None

    except Exception as e_outer:
        print(f"      [T-{thread_id}] CRITICAL THREAD ERROR for {plasmid_id_from_main_page} ({plasmid_page_url}): {type(e_outer).__name__} - {e_outer}")
        # traceback.print_exc() # Can be very verbose in parallel, enable if needed for one thread
        return None
    finally:
        if driver:
            try:
                driver.quit()
            except Exception as e_quit:
                print(f"      [T-{thread_id}] Error quitting driver for {plasmid_id_from_main_page}: {e_quit}")
        # print(f"    [T-{thread_id}] END: {plasmid_id_from_main_page}")


print("Cell 4 execution complete. Threaded core scraping logic REVISED.")

--- Cell 4: Core Scraping Logic for Individual Plasmid Pages (Threaded) - Revised ---
Cell 4 execution complete. Threaded core scraping logic REVISED.


In [32]:
# @title Cell 5: Main Execution Block (Parallelized)
import time
import threading # For getting thread ID for logging, if needed

print("--- Cell 5: Main Execution Block (Parallelized) ---")

def run_scraper_parallel():
    overall_start_time = time.time()

    # Access configuration from Cell 2
    target_url = TARGET_URL
    drive_download_path = DRIVE_DOWNLOAD_PATH
    max_workers_links = MAX_WORKERS_LINK_COLLECTION
    max_workers_downloads = MAX_WORKERS_DOWNLOADS
    max_plasmids_to_process = MAX_PLASMIDS_TO_PROCESS_OVERALL
    download_item_delay = DOWNLOAD_ITEM_DELAY_SECONDS
    # CONFIG dictionary is also globally available from Cell 2

    print(f"--- Starting Addgene GenBank File Scraper (Parallel Version) ---")
    print(f"Target Page: {target_url}")
    print(f"Download Location: {drive_download_path}")
    print(f"Max Plasmid Link Collection Workers: {max_workers_links}")
    print(f"Max File Download Workers: {max_workers_downloads}")
    print(f"Max Plasmids to Process: {'All' if max_plasmids_to_process == 0 else max_plasmids_to_process}")
    print("---------------------------------------------")

    if not os.path.exists(drive_download_path):
        try:
            os.makedirs(drive_download_path, exist_ok=True)
            print(f"Created download directory: {drive_download_path}")
        except Exception as e_dir:
            print(f"CRITICAL ERROR: Could not create download directory {drive_download_path}: {e_dir}")
            return

    # --- Phase 1: Collect Plasmid List from Main Page (Single Driver) ---
    main_page_driver = None
    plasmid_data_list_from_main_page = []
    try:
        print("\n--- Phase 1a: Fetching initial list of plasmids from main page ---")
        main_page_scan_start_time = time.time()
        main_page_driver = setup_driver() # Use the setup_driver from Cell 1
        if not main_page_driver:
            print("CRITICAL ERROR: Could not set up main WebDriver for initial page scan. Exiting.")
            return

        # find_plasmid_data_on_main_page_selenium is from Cell 3, expects (driver, url, config)
        plasmid_data_list_from_main_page = find_plasmid_data_on_main_page_selenium(main_page_driver, target_url, CONFIG)
        main_page_scan_end_time = time.time()
        print(f"Initial plasmid list scan took: {main_page_scan_end_time - main_page_scan_start_time:.2f} seconds.")
        print(f"Found {len(plasmid_data_list_from_main_page)} potential plasmids on the main page.")

    except Exception as e_main_scan:
        print(f"CRITICAL ERROR during initial main page scan: {e_main_scan}")
        traceback.print_exc()
    finally:
        if main_page_driver:
            main_page_driver.quit()
            print("Main page driver quit.")

    if not plasmid_data_list_from_main_page:
        print("\nNo plasmid data found from main page. Exiting scraper.")
        return

    # Apply MAX_PLASMIDS_TO_PROCESS_OVERALL limit
    if max_plasmids_to_process > 0:
        plasmid_data_to_process_links = plasmid_data_list_from_main_page[:max_plasmids_to_process]
        print(f"\nLimiting link collection to the first {len(plasmid_data_to_process_links)} plasmids based on MAX_PLASMIDS_TO_PROCESS_OVERALL.")
    else:
        plasmid_data_to_process_links = plasmid_data_list_from_main_page

    # --- Phase 1b: Collect GenBank Download Links in Parallel ---
    genbank_data_to_download = []
    failed_to_find_gb_link_count = 0
    processed_for_links_count = 0

    print(f"\n--- Phase 1b: Collecting GenBank Download Links in Parallel for {len(plasmid_data_to_process_links)} plasmids ---")
    phase1b_start_time = time.time()

    # get_genbank_download_link_from_plasmid_page_selenium_threaded is from Cell 4
    # It expects (plasmid_info_dict, config_dict)
    # We use a lambda to pass the constant CONFIG to each thread task.
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers_links) as executor:
        future_to_plasmid = {
            executor.submit(get_genbank_download_link_from_plasmid_page_selenium_threaded, p_data, CONFIG): p_data
            for p_data in plasmid_data_to_process_links
        }
        for future in concurrent.futures.as_completed(future_to_plasmid):
            plasmid_data_item = future_to_plasmid[future]
            processed_for_links_count += 1
            try:
                result = future.result() # Get the result from the thread
                if result and result.get('download_url'):
                    genbank_data_to_download.append(result)
                    # Minimal print here, detailed print was inside the thread
                    # print(f"  Successfully queued for download: {result['plasmid_name']} (ID: {result['plasmid_id']})")
                else:
                    failed_to_find_gb_link_count += 1
                    print(f"  WARNING: No GenBank link found or error for plasmid: {plasmid_data_item.get('name', 'N/A')} (ID: {plasmid_data_item.get('id', 'N/A')})")
            except Exception as exc:
                failed_to_find_gb_link_count += 1
                print(f"  CRITICAL ERROR processing plasmid {plasmid_data_item.get('name', 'N/A')} (ID: {plasmid_data_item.get('id', 'N/A')}) for link: {exc}")
                traceback.print_exc()
            finally:
                 # Progress update
                 if processed_for_links_count % (max(1, len(plasmid_data_to_process_links)//10)) == 0 or processed_for_links_count == len(plasmid_data_to_process_links):
                    print(f"  Link collection progress: {processed_for_links_count}/{len(plasmid_data_to_process_links)} plasmids processed for links.")


    phase1b_end_time = time.time()
    print(f"\n--- Link Collection Phase (1b) Summary ---")
    print(f"Time taken for parallel link collection: {phase1b_end_time - phase1b_start_time:.2f} seconds.")
    print(f"Attempted to find links for {processed_for_links_count} plasmid pages.")
    print(f"Successfully found and queued {len(genbank_data_to_download)} GenBank download links.")
    print(f"Failed to find/process GenBank links for: {failed_to_find_gb_link_count} plasmids.")

    if not genbank_data_to_download:
        print("\nNo GenBank links were collected. Nothing to download.")
        overall_end_time = time.time()
        print(f"\n-----------------------------------\")")
        print(f"Scraping process complete. Total script duration: {overall_end_time - overall_start_time:.2f} seconds.")
        return

    # --- Phase 2: Download Files in Parallel ---
    print(f"\n\n--- Phase 2: Downloading {len(genbank_data_to_download)} Collected GenBank Files in Parallel ---")
    phase2_start_time = time.time()
    downloaded_files_count = 0
    failed_to_download_count = 0
    downloaded_files_list = []
    failed_download_details = []

    # download_file_http is from Cell 3
    # It expects (url, folder_path, plasmid_name, plasmid_id, sequence_id)
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers_downloads) as executor:
        future_to_download = {
            executor.submit(download_file_http,
                            data_to_dl['download_url'],
                            drive_download_path, # Global from Cell 2
                            data_to_dl['plasmid_name'],
                            data_to_dl['plasmid_id'],
                            data_to_dl['sequence_id']): data_to_dl
            for data_to_dl in genbank_data_to_download
        }

        processed_downloads = 0
        for future in concurrent.futures.as_completed(future_to_download):
            data_dl_item = future_to_download[future]
            processed_downloads +=1
            try:
                file_path_result = future.result()
                if file_path_result:
                    downloaded_files_count += 1
                    downloaded_files_list.append(file_path_result)
                    # print(f"  Downloaded: {os.path.basename(file_path_result)}") # download_file_http now prints success
                else:
                    failed_to_download_count += 1
                    failed_download_details.append(data_dl_item)
            except Exception as exc_dl:
                failed_to_download_count += 1
                failed_download_details.append({**data_dl_item, 'reason': str(exc_dl)})
                print(f"  CRITICAL ERROR during download task for {data_dl_item['plasmid_name']}: {exc_dl}")
                traceback.print_exc()

            if download_item_delay > 0 and processed_downloads < len(genbank_data_to_download):
                # This delay is now after each download task *completes*.
                # If many workers, this might not be as effective as per-request delay.
                # For true per-request delay in parallel, it needs to be inside download_file_http
                # or use a more complex rate limiter.
                # For now, a small delay between completions.
                time.sleep(download_item_delay / max_workers_downloads if max_workers_downloads > 0 else download_item_delay)


    phase2_end_time = time.time()
    print(f"\nTotal time for parallel download loop: {phase2_end_time - phase2_start_time:.2f} seconds.")
    print(f"\n--- Download Phase Summary ---")

    # --- Final Summary ---
    print("\n\n--- Overall Scraping and Download Summary ---")
    print(f"Attempted to find links for {processed_for_links_count} plasmid pages from {target_url}.")
    print(f"Found {len(genbank_data_to_download)} GenBank download links to attempt.")
    print(f"Successfully downloaded {downloaded_files_count} GenBank files.")
    print(f"Failed to find/process GenBank links for {failed_to_find_gb_link_count} plasmids during collection phase.")
    print(f"Failed to download {failed_to_download_count} files during download phase.")

    if downloaded_files_list:
        print("\nSuccessfully downloaded files:")
        for f_path in downloaded_files_list: print(f"  - {os.path.basename(f_path)}")

    if failed_download_details:
        print("\nFiles/Plasmids with download issues:")
        for item in failed_download_details:
            reason = item.get('reason', 'Download function returned None or did not complete successfully.')
            print(f"  - Plasmid: {item.get('plasmid_name', 'N/A')} (ID: {item.get('plasmid_id','N/A')}), URL: {item.get('download_url','N/A')}, Reason: {reason}")

    overall_end_time = time.time()
    print(f"\n-----------------------------------\")")
    print(f"Scraping process complete. Total script duration: {overall_end_time - overall_start_time:.2f} seconds.")


# --- Main execution check (same as your original) ---
if __name__ == '__main__':
    is_colab_or_ipython = False
    try:
        shell = get_ipython().__class__.__name__
        if shell == 'ZMQInteractiveShell' or shell == 'TerminalInteractiveShell' or 'google.colab' in str(get_ipython()):
            is_colab_or_ipython = True
    except NameError: # get_ipython is not defined
        is_colab_or_ipython = False # Running as a standard Python script

    if is_colab_or_ipython:
        print("INFO: Running in an IPython/Colab environment. Starting PARALLELIZED Selenium scraper.")
        run_scraper_parallel()
    else: # Fallback for non-notebook environments
        print("INFO: Not detected as running in a Colab/IPython notebook environment.")
        print("      To run this script as intended with cell structure, please use Jupyter or Colab.")
        print("      Attempting to run run_scraper_parallel() directly...")
        # Ensure all necessary global variables from previous "cells" would be defined if running this way.
        # This part is tricky if not in a notebook. For Colab, the above check is fine.
        # If you intend to run this as a .py file, you'd combine cells into one script.
        # For now, assuming notebook context for full functionality.
        # run_scraper_parallel() # Uncomment if you want to try running outside notebook, but ensure setup.


print("\nCell 5 execution complete. Parallelized main execution block defined.")
print("To run the scraper, execute Cell 0, then Cell 1, Cell 2, Cell 3, Cell 4, and finally this Cell 5.")

--- Cell 5: Main Execution Block (Parallelized) ---
INFO: Running in an IPython/Colab environment. Starting PARALLELIZED Selenium scraper.
--- Starting Addgene GenBank File Scraper (Parallel Version) ---
Target Page: https://www.addgene.org/browse/article/28196835/
Download Location: /content/drive/My Drive/Addgene_Lavickova2019Maerkl
Max Plasmid Link Collection Workers: 12
Max File Download Workers: 8
Max Plasmids to Process: All
---------------------------------------------

--- Phase 1a: Fetching initial list of plasmids from main page ---

Searching for plasmid data on (Selenium): https://www.addgene.org/browse/article/28196835/
  Navigating to https://www.addgene.org/browse/article/28196835/ with Selenium for initial table load...
  Attempting to find table by ID: plasmids-datatable (PI Page Table)
  Table or its content not found using: PI Page Table (ID: plasmids-datatable) within specified timeout.
  Attempting to find table by ID: DataTables_Table_0 (Article Page Table (Generi

In [ ]:
# @title Cell 7: Annotate GenBank Files with plannotate
# This cell provides functions to annotate downloaded GenBank files using plannotate

import os
import glob
from pathlib import Path
import subprocess
import concurrent.futures

print("--- Cell 7: GenBank Annotation with plannotate ---")

def annotate_genbank_file(input_path, output_dir=None, file_format='genbank'):
    """
    Annotates a single GenBank file using plannotate.
    
    Args:
        input_path (str): Path to the input GenBank file
        output_dir (str): Directory to save annotated file. If None, saves in same directory as input
        file_format (str): Output format ('genbank', 'snapgene', or 'both')
    
    Returns:
        dict: Status information including success/failure and output paths
    """
    try:
        input_path = Path(input_path)
        if not input_path.exists():
            return {
                'success': False,
                'input_file': str(input_path),
                'error': 'Input file does not exist'
            }
        
        # Determine output directory
        if output_dir is None:
            output_dir = input_path.parent / 'annotated'
        else:
            output_dir = Path(output_dir)
        
        output_dir.mkdir(parents=True, exist_ok=True)
        
        # Prepare output filename
        output_file = output_dir / f"{input_path.stem}_annotated{input_path.suffix}"
        
        print(f"  Annotating: {input_path.name}")
        
        # Run plannotate using command line
        # plannotate accepts input file and outputs annotated version
        cmd = [
            'plannotate',
            'batch',
            '-i', str(input_path),
            '-o', str(output_dir),
            '--file_format', file_format
        ]
        
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=300  # 5 minute timeout per file
        )
        
        if result.returncode == 0:
            # Find the output file (plannotate may name it differently)
            possible_outputs = list(output_dir.glob(f"{input_path.stem}*"))
            if possible_outputs:
                print(f"    SUCCESS: Annotated {input_path.name}")
                return {
                    'success': True,
                    'input_file': str(input_path),
                    'output_files': [str(f) for f in possible_outputs],
                    'stdout': result.stdout
                }
            else:
                return {
                    'success': False,
                    'input_file': str(input_path),
                    'error': 'plannotate completed but no output file found',
                    'stdout': result.stdout,
                    'stderr': result.stderr
                }
        else:
            print(f"    ERROR: Failed to annotate {input_path.name}")
            return {
                'success': False,
                'input_file': str(input_path),
                'error': f'plannotate failed with return code {result.returncode}',
                'stderr': result.stderr
            }
            
    except subprocess.TimeoutExpired:
        print(f"    TIMEOUT: Annotation of {input_path.name} exceeded 5 minutes")
        return {
            'success': False,
            'input_file': str(input_path),
            'error': 'Timeout after 5 minutes'
        }
    except Exception as e:
        print(f"    ERROR: Exception while annotating {input_path.name}: {e}")
        return {
            'success': False,
            'input_file': str(input_path),
            'error': str(e)
        }


def annotate_all_genbank_files(input_dir, output_dir=None, file_format='genbank', max_workers=4):
    """
    Annotates all GenBank files in a directory using plannotate with parallel processing.
    
    Args:
        input_dir (str): Directory containing GenBank files to annotate
        output_dir (str): Directory to save annotated files. If None, creates 'annotated' subdirectory
        file_format (str): Output format ('genbank', 'snapgene', or 'both')
        max_workers (int): Maximum number of parallel annotation processes
    
    Returns:
        dict: Summary of annotation results
    """
    input_dir = Path(input_dir)
    
    if not input_dir.exists():
        print(f"ERROR: Input directory does not exist: {input_dir}")
        return {
            'total': 0,
            'successful': 0,
            'failed': 0,
            'results': []
        }
    
    # Find all GenBank files
    gbk_files = list(input_dir.glob('*.gbk')) + list(input_dir.glob('*.gb'))
    
    if not gbk_files:
        print(f"No GenBank files found in {input_dir}")
        return {
            'total': 0,
            'successful': 0,
            'failed': 0,
            'results': []
        }
    
    print(f"\nFound {len(gbk_files)} GenBank files to annotate")
    print(f"Using {max_workers} parallel workers")
    
    # Set output directory
    if output_dir is None:
        output_dir = input_dir / 'annotated'
    else:
        output_dir = Path(output_dir)
    
    print(f"Output directory: {output_dir}\n")
    
    # Process files in parallel
    results = []
    successful = 0
    failed = 0
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_file = {
            executor.submit(annotate_genbank_file, gbk_file, output_dir, file_format): gbk_file
            for gbk_file in gbk_files
        }
        
        for future in concurrent.futures.as_completed(future_to_file):
            result = future.result()
            results.append(result)
            
            if result['success']:
                successful += 1
            else:
                failed += 1
    
    # Print summary
    print("\n" + "="*60)
    print("ANNOTATION SUMMARY")
    print("="*60)
    print(f"Total files processed: {len(gbk_files)}")
    print(f"Successfully annotated: {successful}")
    print(f"Failed: {failed}")
    
    if failed > 0:
        print("\nFailed files:")
        for result in results:
            if not result['success']:
                print(f"  - {Path(result['input_file']).name}: {result.get('error', 'Unknown error')}")
    
    print("="*60 + "\n")
    
    return {
        'total': len(gbk_files),
        'successful': successful,
        'failed': failed,
        'results': results
    }


# Main execution function
def run_plannotate_annotation():
    """
    Main function to annotate GenBank files downloaded by the scraper.
    Uses the DRIVE_DOWNLOAD_PATH from Cell 2 configuration.
    """
    try:
        download_path = DRIVE_DOWNLOAD_PATH  # From Cell 2
        
        print("="*60)
        print("STARTING PLANNOTATE ANNOTATION")
        print("="*60)
        print(f"Input directory: {download_path}")
        
        # Annotate all files
        summary = annotate_all_genbank_files(
            input_dir=download_path,
            output_dir=None,  # Will create 'annotated' subdirectory
            file_format='genbank',
            max_workers=4  # Adjust based on system resources
        )
        
        if summary['successful'] > 0:
            annotated_dir = Path(download_path) / 'annotated'
            print(f"\nAnnotated files saved to: {annotated_dir}")
            print("\nYou can find the fully annotated GenBank files in the 'annotated' subdirectory.")
        
        return summary
        
    except NameError:
        print("ERROR: DRIVE_DOWNLOAD_PATH not defined. Please run Cell 2 (Configuration) first.")
        return None
    except Exception as e:
        print(f"ERROR: An unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()
        return None


print("\nCell 7 execution complete. Plannotate annotation functions defined.")
print("To annotate downloaded GenBank files, run: run_plannotate_annotation()")

In [ ]:
# @title Cell 6: Install plannotate
# Install plannotate for annotating plasmid sequences
# plannotate requires conda/bioconda, so we install condacolab first

print("--- Cell 6: Installing plannotate ---")

# Install condacolab to use conda in Colab
print("Step 1: Installing condacolab to enable conda in Google Colab...")
!pip install -q condacolab

import condacolab
condacolab.install()

print("\nStep 2: Installing plannotate via conda from bioconda channel...")
print("This may take a few minutes...")

# Install plannotate from bioconda
!conda install -c conda-forge -c bioconda plannotate -y

# Setup the plannotate database
print("\nStep 3: Setting up plannotate database...")
!plannotate setupdb

print("\n" + "="*60)
print("plannotate installation complete!")
print("="*60)
print("\nYou can now run Cell 7 to define annotation functions.")
print("Note: After installing conda, you may need to restart the runtime.")
print("      Click 'Runtime > Restart runtime' if prompted.")